<a href="https://colab.research.google.com/github/adinuansah/people-attrition-prediction/blob/main/notebooks/01_build_attrition_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Employee Retention Intelligence — 90-Day Attrition Labels**

Objective: Construct and validate a binary target for voluntary employee exits occurring within 90 calendar days after each eligible employee snapshot.
Data source: Synthetic HRIS dataset generated for this portfolio project.

In [39]:

import pandas as pd

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "adinuansah/people-attrition-prediction/main/"
    "data/synthetic/"
)

employees = pd.read_csv(
    BASE_URL + "employees.csv",
    parse_dates=["hire_date"]
)

exits = pd.read_csv(
    BASE_URL + "employment_exits.csv",
    parse_dates=["exit_date"]
)

snapshots = pd.read_csv(
    BASE_URL + "monthly_snapshots.csv",
    parse_dates=["hire_date", "snapshot_date"]
)

print("Employees:", len(employees))
print("Exits:", len(exits))
print("Active snapshots:", len(snapshots))


Employees: 2000
Exits: 570
Active snapshots: 59065


In [40]:

assert len(employees) == 2000

assert employees["employee_id"].is_unique

assert exits["employee_id"].is_unique

assert not snapshots.duplicated(
    ["employee_id", "snapshot_date"]
).any()

print("PASS: Dataset loaded and key checks completed.")


PASS: Dataset loaded and key checks completed.


In [41]:

# Join exit records to monthly snapshots

labels = snapshots.merge(
    exits[
        ["employee_id", "exit_date", "exit_type"]
    ],
    on="employee_id",
    how="left",
    validate="many_to_one"
)

# Define the future prediction window
labels["prediction_end_date"] = (
    labels["snapshot_date"]
    + pd.Timedelta(days=90)
)

display(
    labels[
        [
            "employee_id",
            "snapshot_date",
            "prediction_end_date",
            "exit_date",
            "exit_type"
        ]
    ].head(10)
)


,employee_id,snapshot_date,prediction_end_date,exit_date,exit_type
0,E00001,2022-01-31,2022-05-01,NaT,NaN
1,E00001,2022-02-28,2022-05-29,NaT,NaN
2,E00001,2022-03-31,2022-06-29,NaT,NaN
3,E00001,2022-04-30,2022-07-29,NaT,NaN
4,E00001,2022-05-31,2022-08-29,NaT,NaN
5,E00001,2022-06-30,2022-09-28,NaT,NaN
6,E00001,2022-07-31,2022-10-29,NaT,NaN
7,E00001,2022-08-31,2022-11-29,NaT,NaN
8,E00001,2022-09-30,2022-12-29,NaT,NaN
9,E00001,2022-10-31,2023-01-29,NaT,NaN


In [42]:

DATA_END_DATE = pd.Timestamp("2025-12-31")

# Does the observation have full 90-day follow-up?
labels["complete_followup"] = (
    labels["prediction_end_date"]
    <= DATA_END_DATE
)

# Did an involuntary departure occur in the window?
labels["competing_exit"] = (
    labels["exit_type"].eq("Involuntary")
    & labels["exit_date"].gt(
        labels["snapshot_date"]
    )
    & labels["exit_date"].le(
        labels["prediction_end_date"]
    )
)

# Our initial binary classification eligibility rule
labels["eligible"] = (
    labels["complete_followup"]
    & ~labels["competing_exit"]
)

print("Total observations:", len(labels))
print("Eligible observations:", labels["eligible"].sum())
print("Excluded observations:", (~labels["eligible"]).sum())


Total observations: 59065
Eligible observations: 54381
Excluded observations: 4684


In [43]:

# Identify voluntary departures within 90 days

voluntary_exit = (
    labels["exit_type"].eq("Voluntary")
    & labels["exit_date"].gt(
        labels["snapshot_date"]
    )
    & labels["exit_date"].le(
        labels["prediction_end_date"]
    )
)

# Create a nullable binary target
labels["voluntary_attrition_next_3m"] = (
    voluntary_exit.astype("Int64")
)

# Remove labels for ineligible observations
labels.loc[
    ~labels["eligible"],
    "voluntary_attrition_next_3m"
] = pd.NA

display(
    labels["voluntary_attrition_next_3m"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="observations")
)


,target,observations
0,0,53213
1,<NA>,4684
2,1,1168


In [44]:

# Validate target construction

target = "voluntary_attrition_next_3m"

# Every eligible observation has a binary label
assert labels.loc[
    labels["eligible"], target
].isin([0, 1]).all()

# Every ineligible observation has a missing label
assert labels.loc[
    ~labels["eligible"], target
].isna().all()

# Every positive label must correspond to
# a voluntary exit in the prediction window
positive = labels.loc[labels[target] == 1]

assert positive["exit_type"].eq("Voluntary").all()

assert (
    positive["exit_date"]
    > positive["snapshot_date"]
).all()

assert (
    positive["exit_date"]
    <= positive["prediction_end_date"]
).all()

print("PASS: All target validation checks passed.")


PASS: All target validation checks passed.



## Target Quality Audit

### Objective
Evaluate the distribution and reliability of the
90-day voluntary attrition target before modeling.

### Analysis Plan
1. Examine class balance and positive event frequency.
2. Quantify excluded observations and their reasons.
3. Analyze target rates across reporting periods.
4. Assess temporal data leakage risks.
5. Design an appropriate train/test split.

### Key Considerations
- Employee-month observations are not independent.
- Prediction windows may overlap.
- Incomplete follow-up must be handled explicitly.
- Future information must not enter model features.


In [45]:

# Target Quality Audit — Class Distribution

target = "voluntary_attrition_next_3m"

# Keep only observations eligible for modeling
eligible_labels = labels.loc[
    labels["eligible"]
].copy()

# Calculate class counts and percentages
class_summary = pd.DataFrame({
    "observations": (
        eligible_labels[target]
        .value_counts()
        .sort_index()
    ),
    "percentage": (
        eligible_labels[target]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
})

display(class_summary)

print("Total eligible observations:", len(eligible_labels))


,observations,percentage
voluntary_attrition_next_3m,,
0,53213,97.85
1,1168,2.15


Total eligible observations: 54381



### Class Distribution — Initial Findings

The labeled dataset contains 54,381 eligible employee-month
observations:

- 53,213 negative observations (97.85%)
- 1,168 positive observations (2.15%)

The target is highly imbalanced, with voluntary departures
within 90 days representing only 2.15% of eligible snapshots.

A majority-class classifier predicting no voluntary departure
would achieve 97.85% accuracy on this dataset while identifying
none of the positive cases.

Consequently, accuracy alone is insufficient for evaluation.
Future modeling will emphasize precision, recall, PR-AUC,
calibration, and comparisons against appropriate baselines.

These figures represent employee-month observations, not
unique employee departures. All data is synthetic.


In [46]:

# Count distinct employees with positive labels

positive_labels = eligible_labels.loc[
    eligible_labels[target] == 1
]

unique_positive_employees = (
    positive_labels["employee_id"].nunique()
)

print(
    "Positive snapshot observations:",
    len(positive_labels)
)

print(
    "Unique employees with positive labels:",
    unique_positive_employees
)

print(
    "Average positive snapshots per employee:",
    round(
        len(positive_labels) / unique_positive_employees,
        2
    )
    if unique_positive_employees > 0
    else 0
)


Positive snapshot observations: 1168
Unique employees with positive labels: 427
Average positive snapshots per employee: 2.74



### Employee-Level vs. Snapshot-Level Outcomes

The dataset contains 1,168 positive employee-month
observations representing 427 unique employees.

Each employee with a positive label contributes an
average of 2.74 positive snapshots, reflecting
overlapping 90-day prediction windows.

These repeated observations are not statistically
independent and must be considered during model
validation and performance evaluation.

The 427 employees represent those with at least one
eligible positive label, not necessarily all voluntary
departures in the full dataset.

All results are based on synthetic workforce data.


In [47]:

# Exclusion Audit — 90-Day Target Eligibility

labels["exclusion_reason"] = "Eligible"

# Full 90-day outcome cannot be observed
labels.loc[
    ~labels["complete_followup"],
    "exclusion_reason"
] = "Incomplete 90-day follow-up"

# Involuntary exit occurs during a complete window
labels.loc[
    labels["complete_followup"]
    & labels["competing_exit"],
    "exclusion_reason"
] = "Involuntary exit within 90 days"

exclusion_summary = (
    labels["exclusion_reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="observations")
)

exclusion_summary["percentage"] = (
    exclusion_summary["observations"]
    / len(labels) * 100
).round(2)

display(exclusion_summary)

print("Total observations:", len(labels))
print("Eligible observations:", labels["eligible"].sum())
print("Excluded observations:", (~labels["eligible"]).sum())


,reason,observations,percentage
0,Eligible,54381,92.07
1,Incomplete 90-day follow-up,4344,7.35
2,Involuntary exit within 90 days,340,0.58


Total observations: 59065
Eligible observations: 54381
Excluded observations: 4684


In [48]:

# Audit overlap between exclusion conditions

print(
    "Incomplete follow-up:",
    (~labels["complete_followup"]).sum()
)

print(
    "Competing involuntary exits:",
    labels["competing_exit"].sum()
)

print(
    "Both conditions:",
    (
        ~labels["complete_followup"]
        & labels["competing_exit"]
    ).sum()
)

assert (
    exclusion_summary["observations"].sum()
    == len(labels)
)

print("PASS: Exclusion categories reconcile.")


Incomplete follow-up: 4344
Competing involuntary exits: 346
Both conditions: 6
PASS: Exclusion categories reconcile.



### Exclusion Audit — Findings

The dataset contains 59,065 employee-month observations.

Of these, 54,381 observations (92.07%) meet the
initial eligibility criteria for 90-day voluntary
attrition classification.

A total of 4,684 observations (7.93%) are excluded:
- 4,344 (7.35%) have incomplete 90-day follow-up.
- 340 (0.58%) have a competing involuntary exit
  within an otherwise complete observation window.

There are 346 competing-exit observations in total,
including 6 that also have incomplete follow-up.

The exclusion categories reconcile without
double-counting.

Limitations:
- The dataset is synthetic.
- Excluding incomplete windows removes recent
  observation periods from fixed-horizon modeling.
- Excluding competing involuntary departures may
  affect the representativeness of the modeled population.


In [49]:

# Inspect eligible snapshot coverage

print(
    "Earliest eligible snapshot:",
    eligible_labels["snapshot_date"].min()
)

print(
    "Latest eligible snapshot:",
    eligible_labels["snapshot_date"].max()
)

# Proposed evaluation boundaries
TRAIN_END = pd.Timestamp("2025-03-31")
TEST_START = pd.Timestamp("2025-07-01")
TEST_END = pd.Timestamp("2025-09-30")

print("\nProposed split:")
print("Training ends:", TRAIN_END.date())
print("Test starts:", TEST_START.date())
print("Test ends:", TEST_END.date())


Earliest eligible snapshot: 2022-01-31 00:00:00
Latest eligible snapshot: 2025-09-30 00:00:00

Proposed split:
Training ends: 2025-03-31
Test starts: 2025-07-01
Test ends: 2025-09-30


## Temporal Train/Test Split

Evaluate model performance on later workforce snapshots
using chronologically separated training and testing periods.

In [50]:

# Temporal Train/Test Split

TRAIN_END = pd.Timestamp("2025-03-31")
TEST_START = pd.Timestamp("2025-07-01")
TEST_END = pd.Timestamp("2025-09-30")

# Training: historical eligible observations
train_df = eligible_labels.loc[
    eligible_labels["snapshot_date"] <= TRAIN_END
].copy()

# Testing: future eligible observations
test_df = eligible_labels.loc[
    eligible_labels["snapshot_date"].between(
        TEST_START, TEST_END
    )
].copy()

print("Training observations:", len(train_df))
print("Testing observations:", len(test_df))


Training observations: 45734
Testing observations: 4370


In [51]:

# Audit class distribution by split

target = "voluntary_attrition_next_3m"

split_summary = pd.DataFrame({
    "dataset": ["Train", "Test"],
    "observations": [
        len(train_df),
        len(test_df)
    ],
    "positive_observations": [
        int(train_df[target].sum()),
        int(test_df[target].sum())
    ],
    "unique_employees": [
        train_df["employee_id"].nunique(),
        test_df["employee_id"].nunique()
    ]
})

split_summary["positive_rate_pct"] = (
    split_summary["positive_observations"]
    / split_summary["observations"] * 100
).round(2)

display(split_summary)


,dataset,observations,positive_observations,unique_employees,positive_rate_pct
0,Train,45734,978,1821,2.14
1,Test,4370,92,1494,2.11


In [52]:

# Temporal validation

assert not train_df.empty
assert not test_df.empty

# Ensure snapshot periods do not overlap
assert train_df["snapshot_date"].max() < (
    test_df["snapshot_date"].min()
)

# Ensure all training labels are observable
# before the start of the future test period
assert train_df["prediction_end_date"].max() < (
    TEST_START
)

# Ensure every test outcome is observed
assert test_df["prediction_end_date"].max() <= (
    DATA_END_DATE
)

# Ensure each subset has unique employee-month records
for dataset in [train_df, test_df]:
    assert not dataset.duplicated(
        ["employee_id", "snapshot_date"]
    ).any()

print("PASS: Temporal split checks completed.")


PASS: Temporal split checks completed.



### Temporal Split — Initial Assessment

The chronological split contains:

| Metric | Training | Test |
|---|---:|---:|
| Observations | 45,734 | 4,370 |
| Positive labels | 978 | 92 |
| Negative labels | 44,756 | 4,278 |
| Positive rate | 2.14% | 2.11% |

The positive-label frequency is similar between
training and testing periods.

The test set contains 92 positive employee-month
observations. This supports an initial evaluation,
but performance estimates may be unstable,
particularly because overlapping prediction windows
create dependence between employee observations.

The test set will be reserved for final model evaluation.

Further checks are required for:
- Feature availability and temporal leakage
- Employee-level dependence
- Feature distribution changes
- Model-selection and preprocessing leakage

All data is synthetic.


In [53]:

# Feature Leakage Audit — Inspect Available Columns

print("TRAINING DATASET COLUMNS")
print("-" * 40)

for column in train_df.columns:
    print(column)

print("\nTEST DATASET COLUMNS")
print("-" * 40)

for column in test_df.columns:
    print(column)

# Check how many unique employees appear
# in each dataset

print("\nUNIQUE EMPLOYEE COUNTS")
print("Train:", train_df["employee_id"].nunique())
print("Test:", test_df["employee_id"].nunique())

# Count distinct employees with positive labels
print("\nUNIQUE POSITIVE EMPLOYEES")

print(
    "Train:",
    train_df.loc[
        train_df[target] == 1,
        "employee_id"
    ].nunique()
)

print(
    "Test:",
    test_df.loc[
        test_df[target] == 1,
        "employee_id"
    ].nunique()
)

# Determine how many employees appear in
# both training and testing periods
shared_employees = (
    set(train_df["employee_id"])
    & set(test_df["employee_id"])
)

print(
    "\nEmployees appearing in both periods:",
    len(shared_employees)
)


TRAINING DATASET COLUMNS
----------------------------------------
employee_id
hire_date
department
job_level
snapshot_date
tenure_years
monthly_salary
exit_date
exit_type
prediction_end_date
complete_followup
competing_exit
eligible
voluntary_attrition_next_3m

TEST DATASET COLUMNS
----------------------------------------
employee_id
hire_date
department
job_level
snapshot_date
tenure_years
monthly_salary
exit_date
exit_type
prediction_end_date
complete_followup
competing_exit
eligible
voluntary_attrition_next_3m

UNIQUE EMPLOYEE COUNTS
Train: 1821
Test: 1494

UNIQUE POSITIVE EMPLOYEES
Train: 351
Test: 59

Employees appearing in both periods: 1347


## Feature Selection and Leakage Controls

Define a restricted set of snapshot-time predictors
and separate them from the future attrition outcome.

In [54]:

# Feature Selection — Initial Modeling Dataset

FEATURE_COLUMNS = [
    "department",
    "job_level",
    "monthly_salary",
    "tenure_years"
]

TARGET_COLUMN = "voluntary_attrition_next_3m"

# Create feature matrices
X_train = train_df[FEATURE_COLUMNS].copy()
X_test = test_df[FEATURE_COLUMNS].copy()

# Create target vectors
y_train = train_df[TARGET_COLUMN].astype(int)
y_test = test_df[TARGET_COLUMN].astype(int)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

display(X_train.head())


X_train: (45734, 4)
y_train: (45734,)
X_test: (4370, 4)
y_test: (4370,)


,department,job_level,monthly_salary,tenure_years
0,Engineering,L4,11388.23,2.48
1,Engineering,L4,11388.23,2.56
2,Engineering,L4,11388.23,2.64
3,Engineering,L4,11388.23,2.72
4,Engineering,L4,11388.23,2.81


In [55]:

# Validate modeling inputs

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert X_train.columns.tolist() == FEATURE_COLUMNS
assert X_test.columns.tolist() == FEATURE_COLUMNS

assert not X_train.isna().any().any()
assert not X_test.isna().any().any()

assert y_train.isin([0, 1]).all()
assert y_test.isin([0, 1]).all()

assert "exit_date" not in X_train.columns
assert "exit_type" not in X_train.columns
assert TARGET_COLUMN not in X_train.columns

print("PASS: Modeling datasets validated.")


PASS: Modeling datasets validated.


## Feature Preprocessing

Categorical variables are one-hot encoded, while numerical
variables are standardized using training-set statistics.

Preprocessing transformations are fitted exclusively on
training data to prevent test-set information leakage.

In [56]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Categorize the feature columns
categorical_features = [
    "department",
    "job_level"
]

numerical_features = [
    "monthly_salary",
    "tenure_years"
]

# Define the preprocessing transformations
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Preprocessing pipeline configured.")


Preprocessing pipeline configured.


In [57]:

# Fit preprocessing on historical training data
X_train_processed = preprocessor.fit_transform(
    X_train
)

# Apply the SAME fitted transformations to test data
X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed test shape:",
    X_test_processed.shape
)

print(
    "\nTransformed feature names:"
)

print(preprocessor.get_feature_names_out())


Processed training shape: (45734, 15)
Processed test shape: (4370, 15)

Transformed feature names:
['department_Data' 'department_Design' 'department_Engineering'
 'department_Finance' 'department_Operations' 'department_People'
 'department_Product' 'job_level_L1' 'job_level_L2' 'job_level_L3'
 'job_level_L4' 'job_level_L5' 'job_level_L6' 'monthly_salary'
 'tenure_years']


In [58]:

import numpy as np

# Train and test must have the same feature count
assert (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
)

# Row counts must remain unchanged
assert (
    X_train_processed.shape[0]
    == len(X_train)
)

assert (
    X_test_processed.shape[0]
    == len(X_test)
)

# No NaN or infinite values
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()

# No target leakage through feature names
feature_names = preprocessor.get_feature_names_out()

assert TARGET_COLUMN not in feature_names
assert "exit_date" not in feature_names
assert "exit_type" not in feature_names

print("PASS: Preprocessing validation completed.")


PASS: Preprocessing validation completed.


## Baseline Model Development

Fit an initial Logistic Regression classifier using
historical training observations and snapshot-time features.

Preprocessing and classification are combined in a
single scikit-learn Pipeline to support reproducibility
and prevent future-data contamination.

The held-out test dataset is reserved for final evaluation.

In [59]:

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

# Reuse the previously defined preprocessor.
# clone() creates a fresh, unfitted copy.
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print(baseline_pipeline)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['department', 'job_level']),
                                                 ('numerical', StandardScaler(),
                                                  ['monthly_salary',
                                                   'tenure_years'])],
                                   verbose_feature_names_out=False)),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


In [60]:

# Train only on historical training observations

baseline_pipeline.fit(
    X_train,
    y_train
)

print("PASS: Logistic Regression model fitted.")


PASS: Logistic Regression model fitted.


In [61]:

import pandas as pd

train_probabilities = (
    baseline_pipeline.predict_proba(
        X_train
    )[:, 1]
)

probability_summary = pd.Series(
    train_probabilities
).describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
)

display(probability_summary)

print(
    "Average predicted probability:",
    round(train_probabilities.mean() * 100, 2),
    "%"
)

print(
    "Actual training positive rate:",
    round(y_train.mean() * 100, 2),
    "%"
)


,0
count,45734.000000
mean,0.021407
std,0.004334
min,0.013209
1%,0.013540
5%,0.016026
50%,0.020947
95%,0.031248
99%,0.034919
max,0.037350


Average predicted probability: 2.14 %
Actual training positive rate: 2.14 %


## Baseline Model Comparison

Compare Logistic Regression with a constant-probability
benchmark before evaluating generalization on historical
validation data.

In [62]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss
)

# Benchmark: predict the training positive rate
dummy_model = DummyClassifier(strategy="prior")

dummy_model.fit(X_train, y_train)

dummy_probabilities = dummy_model.predict_proba(
    X_train
)[:, 1]

print(
    "Constant baseline probability:",
    round(dummy_probabilities[0] * 100, 3),
    "%"
)

print(
    "Logistic Regression mean:",
    round(train_probabilities.mean() * 100, 3),
    "%"
)


Constant baseline probability: 2.138 %
Logistic Regression mean: 2.141 %


In [63]:

# Step 3B — Compare training-set model performance

def evaluate_probabilities(y_true, probabilities):
    return {
        "Average Precision": average_precision_score(
            y_true, probabilities
        ),
        "Brier Score": brier_score_loss(
            y_true, probabilities
        ),
        "Log Loss": log_loss(
            y_true, probabilities
        )
    }

training_comparison = pd.DataFrame({
    "Constant Baseline": evaluate_probabilities(
        y_train, dummy_probabilities
    ),
    "Logistic Regression": evaluate_probabilities(
        y_train, train_probabilities
    )
}).T

display(training_comparison.round(5))


,Average Precision,Brier Score,Log Loss
Constant Baseline,0.02138,0.02093,0.10338
Logistic Regression,0.02556,0.02091,0.10296


## Historical Validation Split

Create chronologically separated development-training
and validation datasets to assess model generalization.

A 90-day separation ensures development-training
outcomes are observed before the validation period.

The final test set remains untouched.

In [64]:

# Step 4A — Historical Validation Split

DEV_TRAIN_END = pd.Timestamp("2024-09-30")

VAL_START = pd.Timestamp("2025-01-01")
VAL_END = pd.Timestamp("2025-03-31")

# Split only the original training dataset
dev_train_df = train_df.loc[
    train_df["snapshot_date"] <= DEV_TRAIN_END
].copy()

val_df = train_df.loc[
    train_df["snapshot_date"].between(
        VAL_START, VAL_END
    )
].copy()

# Create development-training features and targets
X_dev = dev_train_df[FEATURE_COLUMNS].copy()
y_dev = dev_train_df[TARGET_COLUMN].astype(int)

# Create validation features and targets
X_val = val_df[FEATURE_COLUMNS].copy()
y_val = val_df[TARGET_COLUMN].astype(int)

print("Development training:", X_dev.shape)
print("Validation:", X_val.shape)

print("Development positives:", y_dev.sum())
print("Validation positives:", y_val.sum())


Development training: (37496, 4)
Validation: (4153, 4)
Development positives: 796
Validation positives: 87


In [65]:

# Step 4B — Validate Historical Split

assert not dev_train_df.empty
assert not val_df.empty

# Training snapshots precede validation snapshots
assert (
    dev_train_df["snapshot_date"].max()
    < val_df["snapshot_date"].min()
)

# Development outcomes are observed
# before validation starts
assert (
    dev_train_df["prediction_end_date"].max()
    < VAL_START
)

# Validation outcomes are observed
# before final test starts
assert (
    val_df["prediction_end_date"].max()
    < TEST_START
)

# Confirm all selected records are eligible
assert dev_train_df["eligible"].all()
assert val_df["eligible"].all()

# Confirm feature columns match
assert X_dev.columns.tolist() == FEATURE_COLUMNS
assert X_val.columns.tolist() == FEATURE_COLUMNS

print("PASS: Historical validation boundaries verified.")


PASS: Historical validation boundaries verified.


In [66]:

# Step 4C — Historical Validation Population Audit

validation_summary = pd.DataFrame({
    "Dataset": ["Development", "Validation"],

    "Observations": [
        len(y_dev),
        len(y_val)
    ],

    "Positive observations": [
        int(y_dev.sum()),
        int(y_val.sum())
    ],

    "Positive rate (%)": [
        round(y_dev.mean() * 100, 2),
        round(y_val.mean() * 100, 2)
    ],

    "Unique positive employees": [
        dev_train_df.loc[
            y_dev == 1, "employee_id"
        ].nunique(),

        val_df.loc[
            y_val == 1, "employee_id"
        ].nunique()
    ]
})

display(validation_summary)


,Dataset,Observations,Positive observations,Positive rate (%),Unique positive employees
0,Development,37496,796,2.12,293
1,Validation,4153,87,2.09,48


## Out-of-Time Baseline Model Evaluation

Compare a constant-probability benchmark with a
Logistic Regression classifier.

Both models are fitted exclusively on development-training
observations and evaluated on a later validation period.

The final test dataset remains reserved for final evaluation.

In [67]:

# Lesson 3.3 — Step 1
# Fit fresh models on development-training data

from sklearn.base import clone
from sklearn.dummy import DummyClassifier

# Constant-probability benchmark
dev_dummy = DummyClassifier(
    strategy="prior"
)

dev_dummy.fit(
    X_dev,
    y_dev
)

# Fresh Logistic Regression pipeline
# Includes preprocessing fitted only on X_dev
dev_logistic = clone(baseline_pipeline)

dev_logistic.fit(
    X_dev,
    y_dev
)

print("PASS: Both development models fitted.")


PASS: Both development models fitted.


In [68]:

# Lesson 3.3 — Step 2
# Generate validation probabilities

dummy_val_prob = dev_dummy.predict_proba(
    X_val
)[:, 1]

logistic_val_prob = dev_logistic.predict_proba(
    X_val
)[:, 1]

print(
    "Constant baseline probability:",
    round(dummy_val_prob.mean() * 100, 3),
    "%"
)

print(
    "Logistic Regression mean probability:",
    round(logistic_val_prob.mean() * 100, 3),
    "%"
)

print(
    "Actual validation positive rate:",
    round(y_val.mean() * 100, 3),
    "%"
)


Constant baseline probability: 2.123 %
Logistic Regression mean probability: 2.142 %
Actual validation positive rate: 2.095 %


In [69]:

# Lesson 3.3 — Step 3
# Compare out-of-time validation performance

validation_comparison = pd.DataFrame({
    "Constant Baseline": evaluate_probabilities(
        y_val,
        dummy_val_prob
    ),
    "Logistic Regression": evaluate_probabilities(
        y_val,
        logistic_val_prob
    )
}).T

display(validation_comparison.round(5))


,Average Precision,Brier Score,Log Loss
Constant Baseline,0.02095,0.02051,0.10171
Logistic Regression,0.02319,0.02052,0.10179


## Validation Probability Calibration

Assess how predicted attrition probabilities compare
with observed outcome frequencies across model-score
groups in the historical validation period.

Results are descriptive and use synthetic data.

In [70]:

# Lesson 3.4 — Step 1: Validation calibration audit

import pandas as pd

calibration_df = pd.DataFrame({
    "predicted_probability": logistic_val_prob,
    "actual_outcome": y_val.to_numpy()
})

# Group observations into five approximately equal-sized
# groups based on their predicted probabilities.
calibration_df["risk_group"] = pd.qcut(
    calibration_df["predicted_probability"],
    q=5,
    labels=False,
    duplicates="drop"
) + 1

calibration_summary = (
    calibration_df.groupby("risk_group")
    .agg(
        observations=("actual_outcome", "size"),
        positive_observations=("actual_outcome", "sum"),
        mean_predicted_probability=(
            "predicted_probability", "mean"
        ),
        observed_attrition_rate=(
            "actual_outcome", "mean"
        )
    )
    .reset_index()
)

calibration_summary["mean_predicted_pct"] = (
    calibration_summary["mean_predicted_probability"] * 100
).round(2)

calibration_summary["observed_rate_pct"] = (
    calibration_summary["observed_attrition_rate"] * 100
).round(2)

display(
    calibration_summary[
        [
            "risk_group",
            "observations",
            "positive_observations",
            "mean_predicted_pct",
            "observed_rate_pct"
        ]
    ]
)


,risk_group,observations,positive_observations,mean_predicted_pct,observed_rate_pct
0,1,831,14,1.58,1.68
1,2,830,13,1.83,1.57
2,3,831,25,2.03,3.01
3,4,830,17,2.31,2.05
4,5,831,18,2.95,2.17


### Group-Level Calibration Gaps

Compare observed and predicted attrition frequencies
across validation score groups to identify descriptive
patterns of overprediction and underprediction.

The results are interpreted cautiously due to the
small number of positive events and repeated employee
observations.

In [71]:

# Lesson 3.5 — Calibration gap audit

calibration_summary["calibration_gap_pp"] = (
    (
        calibration_summary["observed_attrition_rate"]
        - calibration_summary["mean_predicted_probability"]
    ) * 100
).round(2)

display(
    calibration_summary[
        [
            "risk_group",
            "observations",
            "positive_observations",
            "mean_predicted_pct",
            "observed_rate_pct",
            "calibration_gap_pp"
        ]
    ]
)

print(
    "\nLargest absolute calibration gap:",
    round(
        calibration_summary["calibration_gap_pp"]
        .abs().max(),
        2
    ),
    "percentage points"
)


,risk_group,observations,positive_observations,mean_predicted_pct,observed_rate_pct,calibration_gap_pp
0,1,831,14,1.58,1.68,0.10
1,2,830,13,1.83,1.57,-0.26
2,3,831,25,2.03,3.01,0.98
3,4,830,17,2.31,2.05,-0.26
4,5,831,18,2.95,2.17,-0.79



Largest absolute calibration gap: 0.98 percentage points


## Expanding-Window Historical Validation

Evaluate the stability of model performance across
multiple chronological validation periods.

Each fold uses historical training observations
whose 90-day outcomes are known before validation
begins.

The July–September 2025 final test period remains
excluded from all validation and model-selection steps.

In [72]:

# Lesson 3.6 — Step 1
# Define historical validation windows

validation_folds = [
    {
        "fold": "Fold 1",
        "train_end": "2023-12-31",
        "val_start": "2024-04-01",
        "val_end": "2024-06-30"
    },
    {
        "fold": "Fold 2",
        "train_end": "2024-03-31",
        "val_start": "2024-07-01",
        "val_end": "2024-09-30"
    },
    {
        "fold": "Fold 3",
        "train_end": "2024-09-30",
        "val_start": "2025-01-01",
        "val_end": "2025-03-31"
    }
]

fold_records = []

for fold in validation_folds:

    train_end = pd.Timestamp(fold["train_end"])
    val_start = pd.Timestamp(fold["val_start"])
    val_end = pd.Timestamp(fold["val_end"])

    # Use only eligible observations from
    # our historical training dataset
    fold_train = train_df.loc[
        train_df["snapshot_date"] <= train_end
    ].copy()

    fold_val = train_df.loc[
        train_df["snapshot_date"].between(
            val_start, val_end
        )
    ].copy()

    # Check chronological and outcome boundaries
    assert not fold_train.empty
    assert not fold_val.empty

    assert (
        fold_train["prediction_end_date"].max()
        < val_start
    )

    assert (
        fold_val["prediction_end_date"].max()
        < TEST_START
    )

    fold_records.append({
        "fold": fold["fold"],
        "train_observations": len(fold_train),
        "train_positives": int(
            fold_train[TARGET_COLUMN].sum()
        ),
        "val_observations": len(fold_val),
        "val_positives": int(
            fold_val[TARGET_COLUMN].sum()
        ),
        "val_unique_positive_employees": (
            fold_val.loc[
                fold_val[TARGET_COLUMN] == 1,
                "employee_id"
            ].nunique()
        )
    })

fold_audit = pd.DataFrame(fold_records)

display(fold_audit)

print("PASS: All historical folds passed temporal checks.")


,fold,train_observations,train_positives,val_observations,val_positives,val_unique_positive_employees
0,Fold 1,25896,544,3877,82,46
1,Fold 2,29651,619,3968,95,54
2,Fold 3,37496,796,4153,87,48


PASS: All historical folds passed temporal checks.


In [73]:

# Lesson 3.6 — Step 2
# Calculate positive-label rates

fold_audit["train_positive_rate_pct"] = (
    fold_audit["train_positives"]
    / fold_audit["train_observations"]
    * 100
).round(2)

fold_audit["val_positive_rate_pct"] = (
    fold_audit["val_positives"]
    / fold_audit["val_observations"]
    * 100
).round(2)

display(
    fold_audit[
        [
            "fold",
            "train_observations",
            "train_positive_rate_pct",
            "val_observations",
            "val_positives",
            "val_positive_rate_pct",
            "val_unique_positive_employees"
        ]
    ]
)


,fold,train_observations,train_positive_rate_pct,val_observations,val_positives,val_positive_rate_pct,val_unique_positive_employees
0,Fold 1,25896,2.10,3877,82,2.12,46
1,Fold 2,29651,2.09,3968,95,2.39,54
2,Fold 3,37496,2.12,4153,87,2.09,48


In [74]:

# Lesson 3.7 — Restore checkpoint

display(fold_audit)

print("Historical folds:", len(validation_folds))
print("Model features:", FEATURE_COLUMNS)
print("Final test observations:", len(test_df))


,fold,train_observations,train_positives,val_observations,val_positives,val_unique_positive_employees,train_positive_rate_pct,val_positive_rate_pct
0,Fold 1,25896,544,3877,82,46,2.10,2.12
1,Fold 2,29651,619,3968,95,54,2.09,2.39
2,Fold 3,37496,796,4153,87,48,2.12,2.09


Historical folds: 3
Model features: ['department', 'job_level', 'monthly_salary', 'tenure_years']
Final test observations: 4370


## Expanding-Window Model Evaluation

Compare a constant-probability benchmark and Logistic
Regression across three chronological validation periods.

Refit preprocessing and model parameters independently
within each fold. Preserve the final test period.

In [75]:

# Lesson 3.7 — Step 1
# Fit and evaluate models in each historical fold

from sklearn.base import clone
from sklearn.dummy import DummyClassifier

rolling_results = []

for fold in validation_folds:

    train_end = pd.Timestamp(fold["train_end"])
    val_start = pd.Timestamp(fold["val_start"])
    val_end = pd.Timestamp(fold["val_end"])

    # Historical training and later validation
    fold_train = train_df.loc[
        train_df["snapshot_date"] <= train_end
    ]

    fold_val = train_df.loc[
        train_df["snapshot_date"].between(
            val_start, val_end
        )
    ]

    X_fold_train = fold_train[FEATURE_COLUMNS]
    y_fold_train = fold_train[TARGET_COLUMN].astype(int)

    X_fold_val = fold_val[FEATURE_COLUMNS]
    y_fold_val = fold_val[TARGET_COLUMN].astype(int)

    # Verify label availability before fitting
    assert fold_train["prediction_end_date"].max() < val_start

    # Fresh models for every fold
    models = {
        "Constant Baseline": DummyClassifier(
            strategy="prior"
        ),
        "Logistic Regression": clone(
            baseline_pipeline
        )
    }

    for model_name, model in models.items():

        model.fit(X_fold_train, y_fold_train)

        probabilities = model.predict_proba(
            X_fold_val
        )[:, 1]

        metrics = evaluate_probabilities(
            y_fold_val, probabilities
        )

        rolling_results.append({
            "fold": fold["fold"],
            "model": model_name,
            "observations": len(y_fold_val),
            "positives": int(y_fold_val.sum()),
            **metrics
        })

rolling_comparison = pd.DataFrame(rolling_results)

display(rolling_comparison.round(5))


,fold,model,observations,positives,Average Precision,Brier Score,Log Loss
0,Fold 1,Constant Baseline,3877,82,0.02115,0.02070,0.10248
1,Fold 1,Logistic Regression,3877,82,0.01960,0.02076,0.10368
2,Fold 2,Constant Baseline,3968,95,0.02394,0.02338,0.11323
3,Fold 2,Logistic Regression,3968,95,0.03203,0.02334,0.11241
4,Fold 3,Constant Baseline,4153,87,0.02095,0.02051,0.10171
5,Fold 3,Logistic Regression,4153,87,0.02319,0.02052,0.10179


### Model Stability Across Historical Folds

Compare Logistic Regression against the constant-probability
benchmark across chronological validation periods.

Positive metric improvements indicate that Logistic Regression
outperformed the benchmark; negative values indicate worse
performance.

In [76]:

# Lesson 3.7 — Model stability summary

baseline_scores = (
    rolling_comparison.loc[
        rolling_comparison["model"] == "Constant Baseline"
    ]
    .set_index("fold")
)

logistic_scores = (
    rolling_comparison.loc[
        rolling_comparison["model"] == "Logistic Regression"
    ]
    .set_index("fold")
)

stability_summary = pd.DataFrame({
    "AP improvement": (
        logistic_scores["Average Precision"]
        - baseline_scores["Average Precision"]
    ),
    "Brier improvement": (
        baseline_scores["Brier Score"]
        - logistic_scores["Brier Score"]
    ),
    "Log Loss improvement": (
        baseline_scores["Log Loss"]
        - logistic_scores["Log Loss"]
    )
})

display(stability_summary.round(5))

print("\nFolds where Logistic Regression improved:")
print((stability_summary > 0).sum())


,AP improvement,Brier improvement,Log Loss improvement
fold,,,
Fold 1,-0.00155,-0.00006,-0.00120
Fold 2,0.00809,0.00003,0.00082
Fold 3,0.00224,-0.00001,-0.00008



Folds where Logistic Regression improved:
AP improvement          2
Brier improvement       1
Log Loss improvement    1
dtype: int64



## Historical Model Validation — Findings

### Evaluation Design

A constant-probability baseline and Logistic Regression
classifier were compared across three expanding-window
historical validation periods.

Each fold used a 90-day separation between development
training and validation to prevent future outcome
information from crossing the temporal boundary.

Preprocessing and model parameters were fitted separately
within each fold. The final test period remained untouched.

### Results

| Metric | Folds Favoring Logistic Regression |
|---|---|
| Average Precision | 2 of 3 |
| Brier Score | 1 of 3 |
| Log Loss | 1 of 3 |

Logistic Regression exhibited inconsistent improvements
over the constant-probability benchmark.

The largest Average Precision improvement occurred in
Fold 2, while Fold 1 showed worse performance across
all three metrics.

### Interpretation

The results do not establish consistent incremental
predictive value from the selected workforce features.

The synthetic data-generating process uses a constant
underlying exit hazard and does not deliberately encode
relationships between attrition and the selected features.

Consequently, limited predictive discrimination is
consistent with the simulation design.

### Limitations

- All workforce records and outcomes are synthetic.
- Validation periods contain relatively few distinct
  employees with positive outcomes.
- Employee-month observations are not statistically
  independent because employees appear repeatedly.
- The three validation folds share historical data.
- Model performance does not establish causal relationships
  between workforce characteristics and attrition.
- Results do not support real-world employee-level
  risk scoring or employment decisions.

### Conclusion

The constant-probability model remains the reference
benchmark. Logistic Regression is retained as an
experimental candidate.

Future experiments should separately evaluate a more
realistic synthetic data-generating process with
documented feature-dependent attrition mechanisms.



## Feature Engineering Investigation

### Compensation and Job-Level Structure

Assess the distribution of monthly salary across job
levels in the historical development-training dataset.

The objective is to evaluate whether compensation can
be represented relative to comparable job levels.

Feature transformations must use information available
at prediction time and must not learn parameters from
future validation or test observations.


In [77]:

# Lesson 5.1 — Salary and Job-Level Audit

salary_by_level = (
    dev_train_df.groupby("job_level")
    .agg(
        observations=("monthly_salary", "size"),
        median_salary=("monthly_salary", "median"),
        mean_salary=("monthly_salary", "mean"),
        min_salary=("monthly_salary", "min"),
        max_salary=("monthly_salary", "max")
    )
    .reset_index()
    .sort_values("job_level")
)

display(salary_by_level.round(2))

print(
    "\nMissing salary values:",
    dev_train_df["monthly_salary"].isna().sum()
)

print(
    "Nonpositive salary values:",
    (dev_train_df["monthly_salary"] <= 0).sum()
)

print(
    "Job levels represented:",
    dev_train_df["job_level"].nunique()
)


,job_level,observations,median_salary,mean_salary,min_salary,max_salary
0,L1,3579,3688.52,3711.33,3000.70,4499.41
1,L2,10135,5527.10,5522.56,4504.07,6488.14
2,L3,10920,7698.82,7731.51,6502.55,8990.52
3,L4,7782,10665.66,10705.11,9010.92,12491.03
4,L5,3509,14695.74,14744.35,12526.28,16969.54
5,L6,1571,19561.26,19851.02,17161.04,22919.89



Missing salary values: 0
Nonpositive salary values: 0
Job levels represented: 6


### Job-Level Relative Salary Feature

Construct a salary-to-job-level-median ratio using
reference compensation statistics learned exclusively
from historical training observations.

The transformation preserves the original salary and
job-level features and introduces an additional
relative compensation measure.

All values are based on synthetic HRIS records.

In [78]:

# Lesson 5.2 — Job-Level Salary Ratio Transformer

from sklearn.base import BaseEstimator, TransformerMixin

class SalaryRatioTransformer(
    BaseEstimator, TransformerMixin
):

    def fit(self, X, y=None):

        self.level_medians_ = (
            X.groupby("job_level")["monthly_salary"]
            .median()
        )

        self.overall_median_ = (
            X["monthly_salary"].median()
        )

        return self

    def transform(self, X):

        X_new = X.copy()

        # Reference salary for each job level
        reference_salary = (
            X_new["job_level"]
            .map(self.level_medians_)
            .fillna(self.overall_median_)
        )

        X_new["salary_ratio"] = (
            X_new["monthly_salary"]
            / reference_salary
        )

        return X_new

print("SalaryRatioTransformer defined.")


SalaryRatioTransformer defined.


In [79]:

# Test feature engineering on development data

salary_transformer = SalaryRatioTransformer()

X_dev_enriched = salary_transformer.fit_transform(
    X_dev
)

print(
    "Original features:",
    X_dev.shape[1]
)

print(
    "Enriched features:",
    X_dev_enriched.shape[1]
)

display(
    X_dev_enriched[
        [
            "job_level",
            "monthly_salary",
            "salary_ratio"
        ]
    ].head(10)
)

print("\nSalary ratio summary:")
display(
    X_dev_enriched["salary_ratio"].describe()
)


Original features: 4
Enriched features: 5


,job_level,monthly_salary,salary_ratio
0,L4,11388.23,1.067747
1,L4,11388.23,1.067747
2,L4,11388.23,1.067747
3,L4,11388.23,1.067747
4,L4,11388.23,1.067747
5,L4,11388.23,1.067747
6,L4,11388.23,1.067747
7,L4,11388.23,1.067747
8,L4,11388.23,1.067747
9,L4,11388.23,1.067747



Salary ratio summary:


,salary_ratio
count,37496.000000
mean,1.003303
std,0.097209
min,0.813524
25%,0.921245
50%,1.000000
75%,1.085432
max,1.219842



### Out-of-Time Feature Validation

Apply the salary-ratio transformation to the historical
validation dataset using compensation reference statistics
learned only from development-training observations.

Validate the resulting feature schema, numerical values,
and consistency with training-derived job-level medians.


In [80]:

# Lesson 5.3 — Validate Salary Ratio on Future Data

# IMPORTANT:
# salary_transformer was fitted on X_dev.
# Do not refit it on X_val.

X_val_enriched = salary_transformer.transform(
    X_val
)

# Check feature consistency
assert X_val_enriched.columns.tolist() == (
    X_dev_enriched.columns.tolist()
)

# Check number of observations
assert len(X_val_enriched) == len(X_val)

# Verify valid numerical ratios
assert np.isfinite(
    X_val_enriched["salary_ratio"]
).all()

assert (
    X_val_enriched["salary_ratio"] > 0
).all()

# Verify that training medians were not modified
assert salary_transformer.level_medians_.equals(
    X_dev.groupby("job_level")["monthly_salary"].median()
)

print("PASS: Out-of-time salary feature validated.")

print(
    "\nValidation salary ratio summary:"
)

display(
    X_val_enriched["salary_ratio"].describe()
)

print(
    "\nDevelopment training feature count:",
    X_dev_enriched.shape[1]
)

print(
    "Validation feature count:",
    X_val_enriched.shape[1]
)


PASS: Out-of-time salary feature validated.

Validation salary ratio summary:


,salary_ratio
count,4153.000000
mean,1.002240
std,0.097413
min,0.813524
25%,0.919692
50%,0.999752
75%,1.084652
max,1.219042



Development training feature count: 5
Validation feature count: 5



### Feature-Enriched Logistic Regression

Integrate a training-derived job-level salary-ratio
feature into a scikit-learn modeling pipeline.

The pipeline combines feature engineering, categorical
encoding, numerical scaling, and Logistic Regression.

All learned transformations are fitted only on the
applicable historical training partition.


In [81]:

# Lesson 5.4 — Step 1
# Define feature-enriched Logistic Regression pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_features = [
    "department",
    "job_level"
]

numerical_features_enriched = [
    "monthly_salary",
    "tenure_years",
    "salary_ratio"
]

enriched_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features_enriched
        )
    ],
    remainder="drop"
)

enriched_pipeline = Pipeline(
    steps=[
        (
            "salary_feature",
            SalaryRatioTransformer()
        ),
        (
            "preprocessing",
            enriched_preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("PASS: Enriched pipeline configured.")
print(enriched_pipeline)


PASS: Enriched pipeline configured.
Pipeline(steps=[('salary_feature', SalaryRatioTransformer()),
                ('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['department', 'job_level']),
                                                 ('numerical', StandardScaler(),
                                                  ['monthly_salary',
                                                   'tenure_years',
                                                   'salary_ratio'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


In [82]:

# Lesson 5.4 — Step 2
# Fit only on development-training data

enriched_pipeline.fit(
    X_dev,
    y_dev
)

print("PASS: Enriched Logistic Regression fitted.")

# Inspect the number of features after transformation
transformed_features = (
    enriched_pipeline
    .named_steps["preprocessing"]
    .get_feature_names_out()
)

print(
    "Transformed feature count:",
    len(transformed_features)
)

print("\nTransformed feature names:")
print(transformed_features)


PASS: Enriched Logistic Regression fitted.
Transformed feature count: 16

Transformed feature names:
['categorical__department_Data' 'categorical__department_Design'
 'categorical__department_Engineering' 'categorical__department_Finance'
 'categorical__department_Operations' 'categorical__department_People'
 'categorical__department_Product' 'categorical__job_level_L1'
 'categorical__job_level_L2' 'categorical__job_level_L3'
 'categorical__job_level_L4' 'categorical__job_level_L5'
 'categorical__job_level_L6' 'numerical__monthly_salary'
 'numerical__tenure_years' 'numerical__salary_ratio']


In [84]:

# Lesson 5.4 — Step 3
# Validate the fitted feature-processing sequence

# Extract the fitted pipeline stages
feature_pipeline = enriched_pipeline[:-1]

# Apply the fitted transformations to validation data
X_val_processed_enriched = (
    feature_pipeline.transform(X_val)
)

import numpy as np

# Confirm expected row count
assert (
    X_val_processed_enriched.shape[0]
    == len(X_val)
)

# Confirm expected transformed feature count
assert (
    X_val_processed_enriched.shape[1]
    == len(transformed_features)
)

# Confirm all values are finite
assert np.isfinite(
    X_val_processed_enriched
).all()

print("PASS: Enriched validation features are valid.")

print(
    "Validation feature matrix shape:",
    X_val_processed_enriched.shape
)


PASS: Enriched validation features are valid.
Validation feature matrix shape: (4153, 16)



### Incremental Value of Relative Compensation

Compare the original four-feature Logistic Regression
model with an enriched five-feature model incorporating
job-level-relative salary.

Both models use identical expanding-window historical
validation periods and the same classifier configuration.

Feature engineering, encoding, and scaling are fitted
independently within each training fold.

The final test dataset is excluded from model selection.


In [85]:

# Lesson 5.5 — Compare original and enriched models

from sklearn.base import clone

feature_comparison_results = []

for fold in validation_folds:

    train_end = pd.Timestamp(fold["train_end"])
    val_start = pd.Timestamp(fold["val_start"])
    val_end = pd.Timestamp(fold["val_end"])

    fold_train = train_df.loc[
        train_df["snapshot_date"] <= train_end
    ]

    fold_val = train_df.loc[
        train_df["snapshot_date"].between(
            val_start, val_end
        )
    ]

    # Confirm outcome information is available
    # before the validation period begins.
    assert (
        fold_train["prediction_end_date"].max()
        < val_start
    )

    X_fold_train = fold_train[FEATURE_COLUMNS]
    y_fold_train = (
        fold_train[TARGET_COLUMN].astype(int)
    )

    X_fold_val = fold_val[FEATURE_COLUMNS]
    y_fold_val = (
        fold_val[TARGET_COLUMN].astype(int)
    )

    # Fresh pipelines for each fold
    models = {
        "Original Logistic": clone(baseline_pipeline),
        "Enriched Logistic": clone(enriched_pipeline)
    }

    for model_name, model in models.items():

        model.fit(X_fold_train, y_fold_train)

        probabilities = model.predict_proba(
            X_fold_val
        )[:, 1]

        metrics = evaluate_probabilities(
            y_fold_val, probabilities
        )

        feature_comparison_results.append({
            "fold": fold["fold"],
            "model": model_name,
            "observations": len(y_fold_val),
            "positives": int(y_fold_val.sum()),
            **metrics
        })

feature_comparison = pd.DataFrame(
    feature_comparison_results
)

display(feature_comparison.round(5))


,fold,model,observations,positives,Average Precision,Brier Score,Log Loss
0,Fold 1,Original Logistic,3877,82,0.01960,0.02076,0.10368
1,Fold 1,Enriched Logistic,3877,82,0.01953,0.02077,0.10375
2,Fold 2,Original Logistic,3968,95,0.03203,0.02334,0.11241
3,Fold 2,Enriched Logistic,3968,95,0.03205,0.02334,0.11252
4,Fold 3,Original Logistic,4153,87,0.02319,0.02052,0.10179
5,Fold 3,Enriched Logistic,4153,87,0.02544,0.02052,0.10171


In [86]:

# Lesson 5.5 — Comparison integrity checks

assert len(feature_comparison) == 6

assert (
    feature_comparison.groupby("fold")["model"]
    .nunique() == 2
).all()

assert feature_comparison[
    [
        "Average Precision",
        "Brier Score",
        "Log Loss"
    ]
].notna().all().all()

print("PASS: Feature comparison table validated.")


PASS: Feature comparison table validated.



## Random Forest Model Comparison

Evaluate whether a nonlinear tree-based classifier
provides incremental predictive value over Logistic
Regression and a constant-probability benchmark.

The comparison uses the original four HRIS features
and previously defined expanding-window validation
periods.

The final test dataset remains reserved.


In [87]:

# Lesson 5.6 — Step 1
# Configure a Random Forest classification pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(preprocessor)
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=5,
                min_samples_leaf=30,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Random Forest pipeline configured.")
print(random_forest_pipeline)


Random Forest pipeline configured.
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['department', 'job_level']),
                                                 ('numerical', StandardScaler(),
                                                  ['monthly_salary',
                                                   'tenure_years'])],
                                   verbose_feature_names_out=False)),
                ('classifier',
                 RandomForestClassifier(max_depth=5, min_samples_leaf=30,
                                        n_estimators=200, n_jobs=-1,
                                        random_state=42))])



### Random Forest Historical Validation

Evaluate Random Forest across the same three expanding-window
validation periods used for the existing benchmark models.

Refit preprocessing and model parameters independently within
each historical training window.

Compare Average Precision, Brier Score, and Log Loss without
accessing the reserved final test dataset.


In [88]:

# Lesson 5.6 — Random Forest Historical Validation

rf_results = []

for fold in validation_folds:

    train_end = pd.Timestamp(fold["train_end"])
    val_start = pd.Timestamp(fold["val_start"])
    val_end = pd.Timestamp(fold["val_end"])

    fold_train = train_df.loc[
        train_df["snapshot_date"] <= train_end
    ]

    fold_val = train_df.loc[
        train_df["snapshot_date"].between(
            val_start, val_end
        )
    ]

    # Verify chronological separation
    assert (
        fold_train["prediction_end_date"].max()
        < val_start
    )

    X_fold_train = fold_train[FEATURE_COLUMNS]
    y_fold_train = fold_train[TARGET_COLUMN].astype(int)

    X_fold_val = fold_val[FEATURE_COLUMNS]
    y_fold_val = fold_val[TARGET_COLUMN].astype(int)

    # Fresh Random Forest model for this fold
    rf_model = clone(random_forest_pipeline)

    rf_model.fit(X_fold_train, y_fold_train)

    rf_probabilities = rf_model.predict_proba(
        X_fold_val
    )[:, 1]

    metrics = evaluate_probabilities(
        y_fold_val,
        rf_probabilities
    )

    rf_results.append({
        "fold": fold["fold"],
        "model": "Random Forest",
        "observations": len(y_fold_val),
        "positives": int(y_fold_val.sum()),
        **metrics
    })

rf_comparison = pd.DataFrame(rf_results)

display(rf_comparison.round(5))


,fold,model,observations,positives,Average Precision,Brier Score,Log Loss
0,Fold 1,Random Forest,3877,82,0.01986,0.02075,0.10348
1,Fold 2,Random Forest,3968,95,0.03079,0.02336,0.11291
2,Fold 3,Random Forest,4153,87,0.02277,0.02053,0.10194


In [89]:

# Combine all model results

all_model_results = pd.concat(
    [rolling_comparison, rf_comparison],
    ignore_index=True
)

# Check three models per fold
assert len(all_model_results) == 9

assert (
    all_model_results.groupby("fold")["model"]
    .nunique() == 3
).all()

display(
    all_model_results.sort_values(
        ["fold", "model"]
    ).round(5)
)

print("PASS: Three-model comparison assembled.")


,fold,model,observations,positives,Average Precision,Brier Score,Log Loss
0,Fold 1,Constant Baseline,3877,82,0.02115,0.02070,0.10248
1,Fold 1,Logistic Regression,3877,82,0.01960,0.02076,0.10368
6,Fold 1,Random Forest,3877,82,0.01986,0.02075,0.10348
2,Fold 2,Constant Baseline,3968,95,0.02394,0.02338,0.11323
3,Fold 2,Logistic Regression,3968,95,0.03203,0.02334,0.11241
7,Fold 2,Random Forest,3968,95,0.03079,0.02336,0.11291
4,Fold 3,Constant Baseline,4153,87,0.02095,0.02051,0.10171
5,Fold 3,Logistic Regression,4153,87,0.02319,0.02052,0.10179
8,Fold 3,Random Forest,4153,87,0.02277,0.02053,0.10194


PASS: Three-model comparison assembled.



## Historical Model Selection

### Candidate Models

Three classification approaches were compared across
three chronological validation periods:

1. Constant-probability benchmark
2. Logistic Regression
3. Random Forest

The models used the same four snapshot-time HRIS features.

A separate feature-engineering experiment assessed
job-level-relative compensation but found no consistent
incremental improvement.

### Validation Findings

Neither Logistic Regression nor Random Forest
consistently outperformed the constant-probability
benchmark across historical validation periods.

Random Forest did not demonstrate a reliable advantage
over Logistic Regression.

### Model Selection Decision

The constant-probability classifier is retained as the
reference model for probability forecasting.

The original four-feature Logistic Regression model is
retained as the primary experimental ML candidate because
of its relative simplicity and interpretability.

The enriched salary-ratio model and Random Forest are
not selected for final evaluation.

### Final Evaluation Plan

Both selected approaches will be refitted using the
original historical training period and evaluated once
on the reserved July–September 2025 test observations.

Evaluation will report Average Precision, Brier Score,
Log Loss, and relevant calibration diagnostics.

No additional model or hyperparameter selection will
be performed using final test outcomes.

### Limitations

All workforce data is synthetic, and the departure
simulation does not deliberately encode relationships
between the selected predictors and exit hazard.

Historical validation contains relatively few distinct
positive employees, and repeated employee-month
observations introduce dependence.

Results do not establish the suitability of individual
employee risk scoring or employment decisions.


In [90]:

# Lesson 5.7 — Final Model Selection Check

from sklearn.base import clone
from sklearn.dummy import DummyClassifier

# Selected models (not fitted yet)
final_models = {
    "Constant Baseline": DummyClassifier(
        strategy="prior"
    ),
    "Logistic Regression": clone(
        baseline_pipeline
    )
}

print("Selected models:")
for name in final_models:
    print("-", name)

print("\nSelected features:")
print(FEATURE_COLUMNS)

print("\nTraining period:")
print(
    train_df["snapshot_date"].min(),
    "to",
    train_df["snapshot_date"].max()
)

print("\nReserved test period:")
print(
    test_df["snapshot_date"].min(),
    "to",
    test_df["snapshot_date"].max()
)

# Check original feature set
assert FEATURE_COLUMNS == [
    "department",
    "job_level",
    "monthly_salary",
    "tenure_years"
]

print("\nPASS: Final model selection verified.")


Selected models:
- Constant Baseline
- Logistic Regression

Selected features:
['department', 'job_level', 'monthly_salary', 'tenure_years']

Training period:
2022-01-31 00:00:00 to 2025-03-31 00:00:00

Reserved test period:
2025-07-31 00:00:00 to 2025-09-30 00:00:00

PASS: Final model selection verified.



## Final Held-Out Model Evaluation

Evaluate the selected constant-probability benchmark and
four-feature Logistic Regression classifier on the
reserved July–September 2025 test period.

Model configurations were selected using earlier
chronological validation folds.

All preprocessing and model parameters are fitted
exclusively using historical training observations.

Final test outcomes will not be used for further
model selection or hyperparameter tuning.


In [91]:

# Lesson 5.8 — Fit frozen models

from sklearn.base import clone

fitted_final_models = {}

for name, model in final_models.items():

    fitted_model = clone(model)

    fitted_model.fit(
        X_train,
        y_train
    )

    fitted_final_models[name] = fitted_model

print("PASS: Frozen models fitted on historical training data.")

print("\nFitted models:")
for name in fitted_final_models:
    print("-", name)


PASS: Frozen models fitted on historical training data.

Fitted models:
- Constant Baseline
- Logistic Regression


In [92]:

# Lesson 5.8 — Generate final test probabilities

final_test_probabilities = {}

for name, model in fitted_final_models.items():

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    final_test_probabilities[name] = probabilities

    print(
        name,
        "- Mean predicted probability:",
        round(probabilities.mean() * 100, 3),
        "%"
    )

print(
    "\nActual test positive rate:",
    round(y_test.mean() * 100, 3),
    "%"
)


Constant Baseline - Mean predicted probability: 2.138 %
Logistic Regression - Mean predicted probability: 2.132 %

Actual test positive rate: 2.105 %


In [93]:

# Lesson 5.8 — Final held-out metrics

final_test_results = pd.DataFrame({
    name: evaluate_probabilities(
        y_test, probabilities
    )
    for name, probabilities
    in final_test_probabilities.items()
}).T

display(final_test_results.round(5))

print("\nFinal test observations:", len(y_test))
print("Final positive observations:", int(y_test.sum()))


,Average Precision,Brier Score,Log Loss
Constant Baseline,0.02105,0.02061,0.10211
Logistic Regression,0.02097,0.02065,0.10295



Final test observations: 4370
Final positive observations: 92



## Final Model Evaluation — Findings

### Evaluation Design

Selected model configurations were frozen following
three expanding-window historical validation periods.

The constant-probability benchmark and original
four-feature Logistic Regression model were refitted
using the full historical training dataset.

Final performance was assessed once on 4,370 employee-month
observations from July–September 2025, including
92 positive observations.

### Final Test Results

| Metric | Constant Baseline | Logistic Regression |
|---|---:|---:|
| Average Precision | 0.02105 | 0.02097 |
| Brier Score | 0.02061 | 0.02065 |
| Log Loss | 0.10211 | 0.10295 |

### Interpretation

Logistic Regression did not outperform the constant-
probability benchmark on any of the three reported
metrics.

The observed differences were small, and the findings
are consistent with the historical validation results,
which showed inconsistent model improvements.

### Model Selection Conclusion

The constant-probability benchmark is preferred for
this synthetic forecasting experiment.

The Logistic Regression and Random Forest experiments
demonstrate model-development and evaluation methods,
but do not establish reliable incremental predictive
value.

### Limitations

- All data is synthetic.
- The exit generator uses a constant underlying hazard.
- Selected features have no deliberately encoded
  relationship with departure risk.
- Employee-month observations are repeated and
  statistically dependent.
- The final test set contains only 92 positive
  observations representing 59 distinct employees.
- Reported metrics describe this simulation and
  cannot be generalized to real employee populations.
- Results do not support operational employee-level
  risk scoring or automated employment decisions.
